# Самостоятельная работа 02. Обучение линейной регрессии

Цель: подогнать прямую к облаку точек точно, без итераций, — через частные производные, нормальным уравнением и классом `LinearRegression` — и объяснить, почему все три ответа совпадают.

Правила:

- защищать можно любое число заданий — от одного до всех;
- уйти с занятия раньше времени можно только после защиты хотя бы одного задания;
- баллы ставятся только за защищённые задания: каждое — 1 балл; выполненное, но не защищённое задание даёт 0;
- на защите нужно объяснить любую строку своего кода и при преподавателе изменить его;
- ИИ-помощником можно пользоваться точечно — вспомнить функцию, разобраться с ошибкой; код, который вы не можете объяснить, не засчитывается;
- перед защитой перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и защищаемые задания — их самопроверки должны пройти.

Материалы:

- лекция 04 «Обучение линейной регрессии»: метод наименьших квадратов, частные производные, нормальное уравнение;
- Орельен Жерон, «Прикладное машинное обучение с помощью Scikit-Learn и TensorFlow», гл. 4, стр. 153–158;
- блокнот практического занятия 05 «Обучение линейной регрессии: МНК и градиентный спуск».

| № | Что сделать | Тип | Баллы |
|---|---|---|---|
| 1 | $a$ и $b$ через частные производные | базовое | 1 |
| 2 | Нормальное уравнение и `LinearRegression` | базовое | 1 |

Градиентный спуск для той же прямой вы напишете сами в следующем блокноте — `03-gradient-descent.ipynb`.

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

sys.version, np.__version__, pd.__version__, sklearn.__version__, matplotlib.__version__

## Данные

Служба такси хочет оценивать время поездки по расстоянию. Модель — прямая $\hat{y} = a x + b$: $x$ — расстояние, км; $\hat{y}$ — время поездки, мин; $a$ — сколько минут добавляет каждый километр; $b$ — время на подачу и посадку. Функция ошибки — как на лекции: сумма квадратов остатков $S(a, b) = \sum_{i=1}^{m} (y_i - a x_i - b)^2$ и $\text{MSE} = S / m$.

Ячейки ниже готовы — просто выполните их. Функция `line_mse` считает MSE прямой, функция `is_minimum` нужна самопроверкам: она проверяет, что небольшой сдвиг $a$ или $b$ в любую сторону не уменьшает ошибку.

In [ ]:
def make_trips(n_trips, seed=0):
    """Генерирует поездки такси: расстояние и время в пути.

    Args:
        n_trips: Число поездок.
        seed: Зерно генератора случайных чисел.

    Returns:
        Массивы расстояний, км, и времени поездки, мин.
    """
    rng = np.random.default_rng(seed)
    distance = rng.uniform(0.5, 15, size=n_trips)
    duration = 3 + 2.2 * distance + rng.normal(0, 3, size=n_trips)  # Пробки дают разброс в несколько минут
    return distance, duration


def line_mse(a, b, x, y):
    """Считает MSE прямой y = a·x + b на точках (x, y).

    Args:
        a: Наклон прямой.
        b: Свободный член.
        x: Значения признака.
        y: Целевые значения.

    Returns:
        Среднеквадратичная ошибка.
    """
    return np.mean((y - a * x - b) ** 2)


def is_minimum(a, b, x, y, shift=0.01):
    """Проверяет, что сдвиг a или b на shift в любую сторону не уменьшает MSE.

    Args:
        a: Наклон прямой.
        b: Свободный член.
        x: Значения признака.
        y: Целевые значения.
        shift: Величина сдвига параметра.

    Returns:
        True, если ни один из четырёх сдвигов не уменьшает MSE.
    """
    best_mse = line_mse(a, b, x, y)

    for shift_a, shift_b in [(shift, 0), (-shift, 0), (0, shift), (0, -shift)]:
        if line_mse(a + shift_a, b + shift_b, x, y) < best_mse:
            return False

    return True


distance, duration = make_trips(n_trips=100, seed=42)
distance.shape, duration[:5].round(1)

In [ ]:
plt.scatter(distance, duration, s=12)
plt.xlabel('Расстояние, км')
plt.ylabel('Время поездки, мин')
plt.show()

## Задание 1 (базовое, 1 балл)

На лекции прямую $\hat{y} = a x + b$ провели через три точки (0, 0), (1, 1), (2, 1): записали функцию ошибки $S(a, b)$, взяли частные производные по $a$ и $b$, приравняли их к нулю и решили систему из двух уравнений. Повторите этот путь для всех $m = 100$ поездок — теперь не для конкретных чисел, а в общем виде, через суммы.

**Дано:** массивы `distance` (значения $x_i$) и `duration` (значения $y_i$).

**Получить:** `a_sums` — наклон прямой, `b_sums` — свободный член.

**Как действовать:**

1. Запишите функцию ошибки для $m$ точек: $S(a, b) = \sum_{i=1}^{m} (y_i - a x_i - b)^2$.
2. Найдите частную производную $\partial S / \partial a$. Каждое слагаемое — квадрат выражения, которое зависит от $a$, поэтому дифференцируйте его как сложную функцию (правило цепочки): производная внешней функции $u^2$ равна $2u$, а производная внутренней функции $u = y_i - a x_i - b$ по $a$ равна $-x_i$. Так же найдите $\partial S / \partial b$ — здесь производная внутренней функции по $b$ равна $-1$.
3. Приравняйте обе производные к нулю, разделите на $-2$ и раскройте суммы. Получится система двух линейных уравнений с неизвестными $a$ и $b$, коэффициенты которой — числа $m$, $\sum x_i$, $\sum x_i^2$, $\sum y_i$ и $\sum x_i y_i$.
4. Посчитайте эти суммы на NumPy, соберите матрицу коэффициентов системы размера $2 \times 2$ и вектор правых частей и решите систему.

`LinearRegression`, `np.polyfit` и матрицу признаков здесь не используйте: задание про вывод системы своими руками.

**Что почитать:**

- лекция 04, слайды «Производная по наклону», «Производная по свободному члену» и «Решаем систему уравнений» — тот же вывод для трёх точек;
- [Частная производная](https://ru.wikipedia.org/wiki/Частная_производная) и [Дифференцирование сложной функции](https://ru.wikipedia.org/wiki/Дифференцирование_сложной_функции) (правило цепочки) — Википедия;
- [Система линейных алгебраических уравнений](https://ru.wikipedia.org/wiki/Система_линейных_алгебраических_уравнений) и [метод Крамера](https://ru.wikipedia.org/wiki/Метод_Крамера) — если хотите решить систему $2 \times 2$ формулами;
- [`np.linalg.solve`](https://numpy.org/doc/1.24/reference/generated/numpy.linalg.solve.html) — решает систему по матрице коэффициентов и вектору правых частей; сумма элементов массива — метод `.sum()`.

In [ ]:
a_sums, b_sums = ..., ...  # Ваш код

a_sums, b_sums

In [ ]:
assert is_minimum(a_sums, b_sums, distance, duration), 'Небольшой сдвиг a или b уменьшает ошибку — это не минимум'

'Задание 1: самопроверка пройдена'

## Задание 2 (базовое, 1 балл)

Тот же ответ можно получить без ручного вывода системы: нормальным уравнением, которое записывает условие «все частные производные равны нулю» сразу в матричном виде, и готовым классом scikit-learn.

**Дано:** `distance`, `duration`.

**Получить:** `a_normal`, `b_normal` — параметры по нормальному уравнению на NumPy; `a_sklearn`, `b_sklearn` — параметры из `LinearRegression`.

**Как действовать:**

1. Соберите матрицу признаков $X$ размера $m \times 2$: первый столбец — расстояния, второй — единицы. Столбец единиц отвечает за свободный член: при $\theta = (a, b)$ произведение $X\theta$ даёт $a x_i + b \cdot 1$ для каждой поездки.
2. Вычислите $\hat{\theta} = (X^T X)^{-1} X^T y$: транспонирование — `.T`, матричное умножение — оператор `@`, обратная матрица — `np.linalg.inv`. Первый элемент $\hat{\theta}$ — это $a$, второй — $b$.
3. Обучите `LinearRegression` на тех же данных. Класс ждёт признаки двумерным массивом «образцы × признаки», поэтому одномерный `distance` превратите в столбец; столбец единиц классу не нужен — свободный член он находит сам.
4. Достаньте из обученной модели наклон и свободный член — они хранятся в разных атрибутах.

**Что почитать:**

- лекция 04, слайды «Все точки одной записью», «Нормальное уравнение» и «Подставляем три точки»; книга, стр. 156–158;
- [`np.column_stack`](https://numpy.org/doc/1.24/reference/generated/numpy.column_stack.html), [`np.linalg.inv`](https://numpy.org/doc/1.24/reference/generated/numpy.linalg.inv.html), [`np.reshape`](https://numpy.org/doc/1.24/reference/generated/numpy.reshape.html);
- [`LinearRegression`](https://scikit-learn.org/1.3/modules/generated/sklearn.linear_model.LinearRegression.html) — раздел Attributes: `coef_` и `intercept_`.

In [ ]:
a_normal, b_normal = ..., ...  # Нормальное уравнение на NumPy
a_sklearn, b_sklearn = ..., ...  # LinearRegression

a_normal, b_normal, a_sklearn, b_sklearn

In [ ]:
assert is_minimum(a_normal, b_normal, distance, duration), 'Нормальное уравнение дало не минимум ошибки'
assert np.allclose([a_sklearn, b_sklearn], [a_normal, b_normal]), 'LinearRegression не совпал с нормальным уравнением'

'Задание 2: самопроверка пройдена'

## Сводка и готовность к защите

Ячейка ниже собирает ответы всех способов; она выполнится, когда сделаны оба задания.

Перед тем как звать преподавателя:

- перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и задания, которые будете защищать, — их самопроверки должны пройти;
- убедитесь, что можете объяснить каждую строку своего кода и показать, из какой частной производной получено каждое уравнение системы.

In [ ]:
method_names = ['частные производные', 'нормальное уравнение', 'LinearRegression']
parameters = {
    'a': [a_sums, a_normal, a_sklearn],
    'b': [b_sums, b_normal, b_sklearn],
}
methods = pd.DataFrame(parameters, index=method_names)
methods['MSE'] = [line_mse(a, b, distance, duration) for a, b in zip(methods['a'], methods['b'])]
methods.round(4)